# T03-B — Split-identity and development-integrity diagnostics

**Authorized scope:** train/validation development membership only. This notebook does not inspect any held-out label, feature value, rate, or distribution — only an opaque `PASS`/`FAIL` held-out support status is read and re-recorded. It does not run the T03-C calibration and does not mark T03 accepted.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    DataContractError, PROCESSED_COLUMNS, SOURCE_ROW_ID,
    finalize_artifact_manifest, load_selector, materialize_pandas,
    open_processed_dataset, sha256_file, write_json_new, write_text_new,
)
from src.audit import (
    AuditContractError, DUPLICATE_DEFINITIONS, cross_split_profile_overlap,
    deferred_evidence_record, validate_pre_split_frame,
)
from src.split import (
    SplitContractError, held_out_support_gate, membership_hash,
    support_summary, verify_disjoint_and_complete,
)

NOTEBOOK_PATH = REPO_ROOT / 'notebooks' / 'internal' / 't03b_development_integrity.ipynb'
T03_CONFIG_PATH = REPO_ROOT / 'configs' / 't03_audit.json'
T05_CONFIG_PATH = REPO_ROOT / 'configs' / 't05_split.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'
STAGE = 't03b_development_integrity'
POPULATION = 'train_validation_development_rows'


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

## 1. Verify predecessors: T03-A pre-split and T05 split acceptance

In [2]:
t03_config = json.loads(T03_CONFIG_PATH.read_text(encoding='utf-8'))
t05_config = json.loads(T05_CONFIG_PATH.read_text(encoding='utf-8'))

if t03_config['lifecycle_state'] != 'T03_PRE_SPLIT_COMPLETE':
    raise SplitContractError(f"T03-A is not T03_PRE_SPLIT_COMPLETE (found {t03_config['lifecycle_state']!r})")
if t05_config['lifecycle_state'] != 'T05_SPLIT_ACCEPTED':
    raise SplitContractError(f"T05 is not T05_SPLIT_ACCEPTED (found {t05_config['lifecycle_state']!r})")

t05_run_id = t05_config['lifecycle_state_evidence']['authorizing_run_id']
t05_manifest_path = REPO_ROOT / t05_config['lifecycle_state_evidence']['authorizing_run_manifest']
t05_manifest = json.loads(t05_manifest_path.read_text(encoding='utf-8'))
if not str(t05_manifest['status']).startswith('COMPLETED'):
    raise SplitContractError('T05 authorizing run is not a completed run')

t05_run_root = t05_manifest_path.parent.parent
membership_path = t05_run_root / 'audit' / 'split_membership.csv'
membership = pd.read_csv(membership_path)
observed_hash = membership_hash(membership)
expected_hash = t05_config['lifecycle_state_evidence']['membership_sha256']
if observed_hash != expected_hash:
    raise SplitContractError(f'Split membership hash mismatch: expected {expected_hash}, observed {observed_hash}')

print(f'T03-A: {t03_config["lifecycle_state"]}; T05: {t05_config["lifecycle_state"]} (run {t05_run_id})')
print(f'Split membership reconciled: {observed_hash}')

T03-A: T03_PRE_SPLIT_COMPLETE; T05: T05_SPLIT_ACCEPTED (run t05_split_20260818T073132Z_534290)
Split membership reconciled: cfc3789fdff58cd401aeb6d6e50afe0b26026ef8466ee8e9a7f9c822725f214c


## 2. Load the processed population

In [3]:
selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)
processed_sha256 = selector.payload['processed_sha256']

frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(frame.columns) != PROCESSED_COLUMNS:
    raise AuditContractError('Processed columns changed during materialization')
_ = validate_pre_split_frame(frame, require_zero_based_complete=True)
print(f'Loaded {len(frame):,} rows, processed_sha256={processed_sha256}')

Loaded 13,979,592 rows, processed_sha256=fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c91fd384fa0c0b54fd8


## 3. Immutable T03-B run initialization

In [4]:
RUN_CREATED_AT = utc_now()
RUN_ID = datetime.now(timezone.utc).strftime('t03b_devint_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

try:
    git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    git_head, git_dirty = None, None

run_config = {
    'run_id': RUN_ID, 'created_at_utc': RUN_CREATED_AT, 'stage': STAGE,
    'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH),
    'processed_sha256': processed_sha256,
    'authorizing_t05_run_id': t05_run_id,
    'split_membership_sha256': observed_hash,
    'git_head': git_head, 'git_dirty': git_dirty,
}
write_json_new(RUN_ROOT, 'audit/run_config.json', run_config)
print(f'Run: {RUN_ID}')

Run: t03b_devint_20260818T073803Z_243755


## 4. T03-B.1 — DP-01 / HG-07 hard gate, re-verified independently

T05 already proved disjointness and full row accounting when it built the split; this re-derives the same proof independently, in this run, rather than trusting the earlier JSON record blindly.

In [5]:
disjoint_evidence = verify_disjoint_and_complete(membership, frame[SOURCE_ROW_ID])
if not disjoint_evidence['complete'] or any(disjoint_evidence['pairwise_overlaps'].values()):
    raise SplitContractError('DP-01/HG-07 hard gate failed on independent re-verification')
write_json_new(RUN_ROOT, 'audit/dp01_hg07_reverification.json', {'run_id': RUN_ID, **disjoint_evidence})
print('DP-01/HG-07 PASS:', disjoint_evidence['complete'], disjoint_evidence['counts'])

DP-01/HG-07 PASS: True {'train': 9785714, 'validation': 2096938, 'held_out': 2096940}


## 5. T03-B.2 — DP-06 cross-split profile overlap (train vs. validation only)

For each of DP-02 through DP-05, how many value profiles appear in **both** train and validation? This is descriptive, not a leakage failure by itself — leakage would require the same `_source_row_id` in two splits, which section 4 already ruled out.

In [6]:
from src.data import FEATURE_COLUMNS, TREATMENT_COLUMN, PRIMARY_OUTCOME, SECONDARY_OUTCOME, AUDIT_ONLY_COLUMN

optional = tuple(name for name in (SECONDARY_OUTCOME, AUDIT_ONLY_COLUMN) if name in frame.columns)
dp06_definitions = {
    'DP-02': FEATURE_COLUMNS + (TREATMENT_COLUMN, PRIMARY_OUTCOME) + optional,
    **DUPLICATE_DEFINITIONS,
}
dp06_rows = [
    cross_split_profile_overlap(frame, membership, definition_id=definition, columns=columns)
    for definition, columns in dp06_definitions.items()
]
dp06_report = pd.DataFrame(dp06_rows)
write_text_new(RUN_ROOT, 'audit/dp06_cross_split_overlap.csv', dp06_report.to_csv(index=False, lineterminator='\n'))
dp06_report

,definition_id,columns,splits_compared,rows_compared,overlapping_profile_count,rows_in_overlapping_profiles,interpretation,row_removal_authorized
0,DP-06-DP-02,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...","[train, validation]",11882652,242669,584442,cross-split value-profile repetition; not row ...,False
1,DP-06-DP-03,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...","[train, validation]",11882652,308463,768332,cross-split value-profile repetition; not row ...,False
2,DP-06-DP-04,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...","[train, validation]",11882652,244745,589279,cross-split value-profile repetition; not row ...,False
3,DP-06-DP-05,"[f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, ...","[train, validation]",11882652,244692,589151,cross-split value-profile repetition; not row ...,False


## 6. T03-B.3 — Held-out DP-06 stays sealed

In [7]:
dp06_sealed = deferred_evidence_record('DP-06', held_out_sealed=True)
write_json_new(RUN_ROOT, 'audit/dp06_heldout_sealed.json', {'run_id': RUN_ID, **dp06_sealed})
dp06_sealed

{'audit_id': 'DP-06',
 'method_status': 'ACCEPTED',
 'execution_state': 'SEALED_DEFERRED_BY_TEST_ISOLATION',
 'evidence_status': None,
 'required_action': None,
 'deferred_to': 'T17'}

## 7. T03-B.4 — ED-02 train/validation support, opaque held-out gate

In [8]:
train_support = support_summary(frame, membership, 'train')
validation_support = support_summary(frame, membership, 'validation')
heldout_gate_status = held_out_support_gate(frame, membership)

write_json_new(RUN_ROOT, 'audit/ed02_train_support.json', train_support)
write_json_new(RUN_ROOT, 'audit/ed02_validation_support.json', validation_support)
write_json_new(RUN_ROOT, 'audit/heldout_support_gate.json', {'run_id': RUN_ID, 'status': heldout_gate_status})

print('Train support:', train_support)
print('Validation support:', validation_support)
print('Held-out support gate (opaque):', heldout_gate_status)

Train support: {'split': 'train', 'n': 9785714, 'counts': {'T=0,Y=0': 1465012, 'T=0,Y=1': 2844, 'T=1,Y=0': 8292160, 'T=1,Y=1': 25698}}
Validation support: {'split': 'validation', 'n': 2096938, 'counts': {'T=0,Y=0': 313931, 'T=0,Y=1': 610, 'T=1,Y=0': 1776891, 'T=1,Y=1': 5506}}
Held-out support gate (opaque): PASS


## 8. T03-B.5 — Close the run and record status

In [9]:
audit_rows = pd.DataFrame([
    {'run_id': RUN_ID, 'audit_id': 'DP-01', 'evidence_class': 'HARD_GATE', 'evidence_status': 'PASS', 'required_action': 'PASS', 'execution_state': 'EXECUTED'},
    {'run_id': RUN_ID, 'audit_id': 'ED-02', 'evidence_class': 'EMPIRICAL_DIAGNOSTIC', 'evidence_status': 'INFO', 'required_action': 'PASS', 'execution_state': 'EXECUTED'},
    {'run_id': RUN_ID, 'audit_id': 'DP-06-train-validation', 'evidence_class': 'EMPIRICAL_DIAGNOSTIC', 'evidence_status': 'INFO', 'required_action': 'PASS', 'execution_state': 'EXECUTED'},
    {'run_id': RUN_ID, 'audit_id': 'DP-06-held-out', 'evidence_class': 'EMPIRICAL_DIAGNOSTIC', 'evidence_status': None, 'required_action': None, 'execution_state': 'SEALED_DEFERRED_BY_TEST_ISOLATION'},
])
write_text_new(RUN_ROOT, 'audit/audit_summary.csv', audit_rows.to_csv(index=False, lineterminator='\n'))

summary = {
    'run_id': RUN_ID, 'status': 'COMPLETED_T03B_DEVELOPMENT_INTEGRITY',
    'task_lifecycle_state': 'T03_DEVELOPMENT_INTEGRITY_COMPLETE',
    'dp01_hg07_pass': True,
    'heldout_support_gate': heldout_gate_status,
    'heldout_dp06_sealed': True,
    'held_out_labels_or_features_accessed': False,
    'rows_removed': 0,
}
write_json_new(RUN_ROOT, 'audit/t03b_summary.json', summary)

finalize_artifact_manifest(
    RUN_ROOT, run_id=RUN_ID, final_status=summary['status'], created_at_utc=utc_now(),
    stage=STAGE, population=POPULATION,
    external_artifacts=[
        {'path': selector.processed_path.name, 'role': 'manifest_selected_processed_derivative', 'sha256': processed_sha256, 'status': 'PASS'},
        {'path': 'notebooks/internal/t03b_development_integrity.ipynb#sources', 'role': 'human_readable_protocol_source', 'sha256': notebook_source_sha256(NOTEBOOK_PATH), 'status': 'PASS'},
        {'path': 'src/audit.py', 'role': 'reusable_t03_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'audit.py'), 'status': 'PASS'},
    ],
)
summary

{'run_id': 't03b_devint_20260818T073803Z_243755',
 'status': 'COMPLETED_T03B_DEVELOPMENT_INTEGRITY',
 'task_lifecycle_state': 'T03_DEVELOPMENT_INTEGRITY_COMPLETE',
 'dp01_hg07_pass': True,
 'heldout_support_gate': 'PASS',
 'heldout_dp06_sealed': True,
 'held_out_labels_or_features_accessed': False,
 'rows_removed': 0}